# Day 09. Exercise 01
# Gridsearch

## 0. Imports

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, ParameterGrid, cross_val_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from tqdm.notebook import tqdm

## 1. Preprocessing

1. Read the file [`day-of-week-not-scaled.csv`](https://drive.google.com/file/d/1AlGvsJDSzPT_70caausx8bFuupIEZkfh/view?usp=sharing). It is similar to the one from the previous exercise, but this time we did not scale continuous features (we are not going to use logreg anymore). Don't forget to enrich the table with the 'dayofweek' column from the previous day's .csv-file.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [2]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv').reset_index()
df_for_merge = pd.read_csv('../data/dayofweek.csv').reset_index()
df['dayofweek'] = df_for_merge['dayofweek']

df = df.drop(columns='index', axis=1)

X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1348, 43), (338, 43), (1348,), (338,))

## 2. SVM gridsearch

1. Using `GridSearchCV` try different parameters of kernel (`linear`, `rbf`, `sigmoid`), C (`0.01`, `0.1`, `1`, `1.5`, `5`, `10`), gamma (`scale`, `auto`), class_weight (`balanced`, `None`) use `random_state=21` and `probability=True` and get the best combination of them in terms of accuracy.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`. Check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [3]:
%%time
svc = SVC(random_state=21, probability=True)

grid_space = {
    "kernel": ['linear', 'rbf', 'sigmoid'],
    "C": [0.01, 0.1, 1, 1.5, 5, 10],
    "gamma": ['scale', 'auto'],
    "class_weight": [None, 'balanced']
}

grid = GridSearchCV(
    svc,
    param_grid=grid_space,
    scoring='accuracy'
)

grid.fit(X_train, y_train)
print(f'Лучшая точность:{grid.best_score_}, при параметрах {grid.best_params_}')

Лучшая точность:0.8761090458488228, при параметрах {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf'}
CPU times: user 30min 21s, sys: 939 ms, total: 30min 22s
Wall time: 30min 29s


In [4]:
results = pd.DataFrame(grid.cv_results_).sort_values('rank_test_score')
results.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_class_weight,param_gamma,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
64,0.561459,0.004817,0.018450,0.000564,10.0,None,auto,rbf,"{'C': 10, 'class_weight': None, 'gamma': 'auto...",0.900000,0.848148,0.885185,0.884758,0.862454,0.876109,0.018419,1
70,0.555359,0.004369,0.017784,0.000318,10.0,balanced,auto,rbf,"{'C': 10, 'class_weight': 'balanced', 'gamma':...",0.877778,0.851852,0.862963,0.873606,0.851301,0.863500,0.010870,2
52,0.580474,0.052169,0.023959,0.004816,5.0,None,auto,rbf,"{'C': 5, 'class_weight': None, 'gamma': 'auto'...",0.825926,0.811111,0.818519,0.821561,0.802974,0.816018,0.008116,3
58,0.578617,0.006226,0.021235,0.001093,5.0,balanced,auto,rbf,"{'C': 5, 'class_weight': 'balanced', 'gamma': ...",0.844444,0.785185,0.792593,0.817844,0.802974,0.808608,0.021007,4
69,40.958870,4.741189,0.009601,0.000286,10.0,balanced,auto,linear,"{'C': 10, 'class_weight': 'balanced', 'gamma':...",0.729630,0.700000,0.755556,0.754647,0.665428,0.721052,0.034438,5


## 3. Decision tree

1. Using `GridSearchCV` try different parameters of `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use `random_state=21`.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [5]:
%%time
dct = DecisionTreeClassifier(random_state=21)

grid_space = {
    "max_depth": range(1, 50),
    "class_weight": [None, 'balanced'],
    "criterion": ['entropy','gini']
}

grid = GridSearchCV(
    dct,
    param_grid=grid_space,
    cv=5,
    scoring='accuracy'
)

grid.fit(X_train, y_train)
print(f'Лучшая точность:{grid.best_score_}, при параметрах {grid.best_params_}')

Лучшая точность:0.8731212997384002, при параметрах {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 22}
CPU times: user 7.32 s, sys: 22.7 ms, total: 7.34 s
Wall time: 7.38 s


In [6]:
results = pd.DataFrame(grid.cv_results_).sort_values('rank_test_score')
results.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_class_weight,param_criterion,param_max_depth,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
168,0.005759,0.000132,0.001896,0.000036,balanced,gini,22,"{'class_weight': 'balanced', 'criterion': 'gin...",0.885185,0.862963,0.903704,0.881041,0.832714,0.873121,0.023998,1
167,0.005777,0.000128,0.001869,0.000009,balanced,gini,21,"{'class_weight': 'balanced', 'criterion': 'gin...",0.888889,0.859259,0.903704,0.884758,0.828996,0.873121,0.026300,2
195,0.005811,0.000105,0.001869,0.000007,balanced,gini,49,"{'class_weight': 'balanced', 'criterion': 'gin...",0.888889,0.866667,0.903704,0.873606,0.832714,0.873116,0.023911,3
194,0.005806,0.000114,0.001868,0.000005,balanced,gini,48,"{'class_weight': 'balanced', 'criterion': 'gin...",0.888889,0.866667,0.903704,0.873606,0.832714,0.873116,0.023911,3
169,0.005781,0.000150,0.001886,0.000035,balanced,gini,23,"{'class_weight': 'balanced', 'criterion': 'gin...",0.888889,0.866667,0.903704,0.873606,0.832714,0.873116,0.023911,3


## 4. Random forest

1. Using `GridSearchCV` try different parameters of `n_estimators` (`5`, `10`, `50`, `100`), `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use random_state=21.
2. Create a dataframe from the results of the gridsearch and sort it ascendengly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [7]:
%%time
rfc = RandomForestClassifier(random_state=21)

grid_space = {
    "n_estimators": [5, 10, 50, 100],
    "max_depth": range(1, 50),
    "class_weight": [None, 'balanced'],
    "criterion": ['entropy','gini']
}

grid = GridSearchCV(
    rfc,
    param_grid=grid_space,
    scoring='accuracy',
    verbose=1
)

grid.fit(X_train, y_train)
print(f'Лучшая точность:{grid.best_score_}, при параметрах {grid.best_params_}')

Fitting 5 folds for each of 784 candidates, totalling 3920 fits
Лучшая точность:0.9042902381935839, при параметрах {'class_weight': None, 'criterion': 'gini', 'max_depth': 28, 'n_estimators': 50}
CPU times: user 5min 49s, sys: 506 ms, total: 5min 49s
Wall time: 5min 51s


In [8]:
results = pd.DataFrame(grid.cv_results_).sort_values('rank_test_score')
results.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_class_weight,param_criterion,param_max_depth,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
306,0.102435,0.000622,0.006499,0.000189,None,gini,28,50,"{'class_weight': None, 'criterion': 'gini', 'm...",0.922222,0.900000,0.907407,0.903346,0.888476,0.904290,0.010961,1
319,0.203509,0.001432,0.011040,0.000167,None,gini,31,100,"{'class_weight': None, 'criterion': 'gini', 'm...",0.918519,0.911111,0.900000,0.910781,0.877323,0.903547,0.014380,2
706,0.106091,0.001706,0.006571,0.000192,balanced,gini,30,50,"{'class_weight': 'balanced', 'criterion': 'gin...",0.922222,0.907407,0.881481,0.907063,0.895911,0.902817,0.013554,3
722,0.105153,0.000645,0.006497,0.000043,balanced,gini,34,50,"{'class_weight': 'balanced', 'criterion': 'gin...",0.922222,0.907407,0.892593,0.907063,0.884758,0.902809,0.013010,4
387,0.201965,0.000429,0.010886,0.000088,None,gini,48,100,"{'class_weight': None, 'criterion': 'gini', 'm...",0.914815,0.911111,0.900000,0.903346,0.884758,0.902806,0.010460,5


## 5. Progress bar

Gridsearch can be a quite long process and you may find yourself wondering when it will end.
1. Create a manual gridsearch for the same parameters values of random forest iterating through the list of the possible values and calculating `cross_val_score` for each combination. Try to increase `n_jobs`. The value `cv` for `cross_val_score` is 5.
2. Track the progress using the library `tqdm.notebook`.
3. Create a dataframe from the results of the gridsearch with the columns corresponding to the names of the parameters and `mean_accuracy` and `std_accuracy`.
4. Sort it descendingly by the `mean_accuracy`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [9]:
params = {
    'n_estimators': [5, 10, 50, 100],
    'max_depth': [30, 50, 100],
    'class_weight': [None, 'balanced'],
    'criterion': ['gini', 'entropy'],
    'random_state': [21],
}

params_list = list(ParameterGrid(params))
data = []

for params_model in tqdm(params_list):
    estimator = RandomForestClassifier(**params_model)
    cvs = cross_val_score(estimator, X_train, y_train, cv=5, n_jobs=1)
    info = {**params_model, 'mean_accuracy': cvs.mean(), 'std_accuracy': cvs.std()}
    data.append(info)

  0%|          | 0/48 [00:00<?, ?it/s]

In [10]:
results = pd.DataFrame(data)
results.sort_values('mean_accuracy', ascending=False).head()

,class_weight,criterion,max_depth,n_estimators,random_state,mean_accuracy,std_accuracy
26,balanced,gini,30,50,21,0.902817,0.013554
7,None,gini,50,100,21,0.902806,0.010460
11,None,gini,100,100,21,0.902806,0.010460
3,None,gini,30,100,21,0.902068,0.012843
34,balanced,gini,100,50,21,0.902065,0.014082


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.

In [11]:
rfc = RandomForestClassifier(random_state=21, class_weight = None, criterion = 'gini', max_depth = 28, n_estimators = 50).fit(X_train, y_train)
rfc.score(X_test, y_test)

0.9289940828402367